# Bronze Layer: Raw Seismic Table Definition
This notebook creates the physical Delta table to store the raw JSON payloads directly from the USGS API. No transformations or cleaning rules are applied here to preserve data lineage and historical raw state.

In [0]:
%sql
-- Ensure we are using the correct catalog context
USE CATALOG cr_seismic_analysis;

-- Create the Managed Volume explicitly inside the bronze schema
CREATE VOLUME IF NOT EXISTS bronze.seismic_data_raw
COMMENT 'Bronze volume containg raw information retrieved from the API';

-- Create the new flat Bronze table with resilient string types and full metadata
CREATE TABLE IF NOT EXISTS cr_seismic_analysis.bronze.seismic_raw (
    event_id STRING COMMENT 'Unique earthquake identifier from USGS',
    magnitude STRING COMMENT 'Seismic magnitude value represented as a string',
    place STRING COMMENT 'Textual description of the geographic location',
    event_timestamp_raw STRING COMMENT 'Raw epoch millisecond timestamp from the API source',
    event_type STRING COMMENT 'Type of seismic event (e.g., earthquake, quarry blast)',
    longitude STRING COMMENT 'Geographic coordinate longitude',
    latitude STRING COMMENT 'Geographic coordinate latitude',
    depth STRING COMMENT 'Depth of the seismic event in kilometers',
    source_file STRING COMMENT 'Audit metadata: Name of the raw source JSON file',
    ingestion_timestamp TIMESTAMP COMMENT 'Audit metadata: System timestamp when written to Delta'
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
COMMENT 'Bronze table containing flattened raw seismic events with resilient string types';

In [0]:
%sql
SELECT volume_catalog, volume_schema, volume_name 
FROM cr_seismic_analysis.information_schema.volumes 
WHERE volume_name = 'seismic_data_raw';

In [0]:
%sql

-- Check that your table is ready
SHOW TABLES IN cr_seismic_analysis.bronze;